In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_selection.data_loading import load_split
from src.feature_selection.mutual_information import (
    calculate_mutual_information,
    calculate_feature_mutual_information,
)


In [4]:
# Mutual information must only ever see the train split.
# (Previously this read data/processed/ml_dataset.csv directly,
# which includes validation+test rows -> test leakage.)
X, y = load_split("train", processed_dir=PROJECT_ROOT / "data" / "processed")

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of features:", X.shape[1])


X shape: (1895, 25)
y shape: (1895,)
Number of features: 25


In [5]:
print(X.dtypes)
print("Missing values:", X.isnull().sum().sum())


return_1d           float64
return_5d           float64
return_10d          float64
return_20d          float64
intraday_return     float64
high_low_range      float64
gap                 float64
sma_5               float64
sma_20              float64
sma_60              float64
price_to_sma_5      float64
price_to_sma_20     float64
price_to_sma_60     float64
rsi_14              float64
roc_10              float64
roc_20              float64
macd                float64
macd_signal         float64
macd_hist           float64
volatility_5        float64
volatility_20       float64
atr_14              float64
volume_change_1d    float64
volume_sma_20       float64
volume_ratio_20     float64
dtype: object
Missing values: 0


In [6]:
mi_scores = calculate_mutual_information(
    X,
    y,
    random_state=42,
)

mi_scores


rsi_14              0.070074
volatility_20       0.064134
macd_signal         0.056946
volume_sma_20       0.055255
atr_14              0.050383
price_to_sma_20     0.042641
sma_60              0.040161
price_to_sma_60     0.038265
volatility_5        0.029891
macd                0.026377
return_1d           0.025765
sma_20              0.022646
return_20d          0.020794
roc_20              0.020733
volume_ratio_20     0.019291
sma_5               0.018318
gap                 0.015860
price_to_sma_5      0.014590
roc_10              0.014314
macd_hist           0.014307
return_10d          0.014226
return_5d           0.010298
volume_change_1d    0.001885
high_low_range      0.000000
intraday_return     0.000000
Name: mutual_information, dtype: float64

In [7]:
mi_results = mi_scores.reset_index()
mi_results.columns = ["feature", "mi_score"]

mi_results


,feature,mi_score
0,rsi_14,0.070074
1,volatility_20,0.064134
2,macd_signal,0.056946
3,volume_sma_20,0.055255
4,atr_14,0.050383
5,price_to_sma_20,0.042641
6,sma_60,0.040161
7,price_to_sma_60,0.038265
8,volatility_5,0.029891
9,macd,0.026377


In [8]:
# Reuses the same train-only X loaded above -- do NOT reload
# from ml_dataset.csv here.
mi_matrix = calculate_feature_mutual_information(
    X,
    random_state=42,
)

mi_matrix


,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,sma_20,sma_60,...,roc_20,macd,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,volume_change_1d,volume_sma_20,volume_ratio_20
return_1d,0.000000,0.123459,0.065863,0.053170,0.646476,0.184390,0.205498,0.134822,0.054703,0.000705,...,0.053170,0.000000,0.000510,0.029407,0.133122,0.050404,0.024989,0.122212,0.061600,0.150318
return_5d,0.123459,0.000000,0.362350,0.133865,0.040889,0.036107,0.065457,0.019686,0.020814,0.046237,...,0.133865,0.079196,0.057141,0.255325,0.140954,0.098751,0.042795,0.000000,0.072301,0.095413
return_10d,0.065863,0.362350,0.000000,0.390133,0.036547,0.090082,0.048655,0.128769,0.110519,0.179631,...,0.390133,0.212396,0.102393,0.582723,0.104611,0.172882,0.142541,0.003694,0.174118,0.057390
return_20d,0.053170,0.133865,0.390133,0.000000,0.001533,0.036861,0.061230,0.259545,0.185778,0.271508,...,6.279094,0.651503,0.403877,0.247327,0.148561,0.325734,0.266493,0.000000,0.267052,0.072073
intraday_return,0.646476,0.040889,0.036547,0.001533,0.000000,0.541085,0.029510,0.127586,0.092206,0.049659,...,0.001415,0.004067,0.000000,0.000000,0.067046,0.039757,0.033459,0.037704,0.016560,0.114446
high_low_range,0.184390,0.036107,0.090082,0.036861,0.541085,0.000000,0.002116,0.180059,0.132830,0.079385,...,0.036910,0.077528,0.082985,0.080900,0.144337,0.124975,0.073518,0.078496,0.109608,0.142226
gap,0.205498,0.065457,0.048655,0.061230,0.029510,0.002116,0.000000,0.375164,0.275341,0.201939,...,0.062323,0.042535,0.068990,0.001562,0.041916,0.067963,0.155837,0.040018,0.126864,0.075025
sma_5,0.134822,0.019686,0.128769,0.259545,0.127586,0.180059,0.375164,0.000000,2.473325,2.273623,...,0.259361,1.057422,1.120298,0.498408,0.239622,0.836801,1.625898,0.000000,1.357332,0.035572
sma_20,0.054703,0.020814,0.110519,0.185778,0.092206,0.132830,0.275341,2.473325,0.000000,2.772745,...,0.185828,0.906980,1.117407,0.529053,0.243701,0.904783,1.816327,0.000000,1.491676,0.052796
sma_60,0.000705,0.046237,0.179631,0.271508,0.049659,0.079385,0.201939,2.273623,2.772745,0.000000,...,0.271592,0.966495,1.110351,0.585129,0.251665,0.925035,1.834135,0.020886,1.619018,0.067996


In [9]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "filter_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

mi_matrix.to_csv(OUTPUT_DIR / "feature_mi_matrix.csv")


In [10]:
mi_results.to_csv(
    OUTPUT_DIR / "feature_target_mi.csv",
    index=False,
)
